# Cadrille — Google Colab

Supports:
- **Eval**: run inference + metrics on a checkpoint (free T4 / Pro A100)
- **SFT smoke test**: 50-step training run to verify the pipeline
- **RL fine-tuning**: Dr. CPPO on H100 (Pro+ recommended)
- **Monitoring**: sync W&B offline runs

> Mount Google Drive first — data and checkpoints survive session restarts there.

In [ ]:
# ── [1] Mount Drive ──────────────────────────────────────────────────────────
# All data and checkpoints live here — they survive session restarts.
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT  = '/content/drive/MyDrive/cadrille'   # adjust if needed
DRIVE_DATA  = f'{DRIVE_ROOT}/data'
DRIVE_CKPTS = f'{DRIVE_ROOT}/checkpoints'

import os
os.makedirs(DRIVE_DATA,  exist_ok=True)
os.makedirs(DRIVE_CKPTS, exist_ok=True)
print('Drive mounted. Paths:', DRIVE_DATA, DRIVE_CKPTS)

In [ ]:
# ── [2] Clone repo ───────────────────────────────────────────────────────────
import os

REPO = '/content/cadrille'
if not os.path.exists(REPO):
    !git clone https://github.com/miachen0401/cadrille.git {REPO}
else:
    !git -C {REPO} pull   # update to latest

%cd {REPO}

In [ ]:
# ── [3] Install dependencies ─────────────────────────────────────────────────
# Run once per session (takes ~5 min on first run, cached on restart).
# open3d: standard pip version works on Colab (headless rendering via osmesa).
# cadquery: lightweight pip version — avoids the full source build in Docker.

!pip install -q \
    "transformers==4.50.3" \
    "accelerate==0.34.2" \
    "qwen-vl-utils==0.0.10" \
    "bitsandbytes>=0.43.0" \
    "flash-attn==2.7.2.post1" --no-build-isolation \
    "cadquery" \
    "cadquery-ocp==7.7.2" \
    "trimesh==4.5.3" \
    "open3d" \
    "scipy==1.14.1" \
    "wandb" \
    "tqdm" \
    "pyyaml"

# Verify GPU
import torch
p = torch.cuda.get_device_properties(0)
print(f'GPU: {p.name}  {p.total_memory // 1024**3} GB  |  CUDA {torch.version.cuda}')

In [ ]:
# ── [4] Link data & checkpoints from Drive ───────────────────────────────────
import os

if not os.path.exists('data'):
    os.symlink(DRIVE_DATA, 'data')

if not os.path.exists('checkpoints'):
    os.symlink(DRIVE_CKPTS, 'checkpoints')

print('data/      →', os.listdir('data') or '(empty)')
print('checkpoints/ →', os.listdir('checkpoints') or '(empty)')

In [ ]:
# ── [5] Download training data (run once, data persists on Drive) ─────────────
# DeepCAD + Fusion360 meshes from HuggingFace.
# Skip this cell if data/ already has the mesh folders.

import os

if not os.path.exists('data/deepcad_train_mesh'):
    # ~2 GB — DeepCAD training split (STL meshes)
    !huggingface-cli download maksimko123/cadrille-data \
        --repo-type dataset \
        --local-dir data/

print('Data ready:')
!ls data/

In [ ]:
# ── [6] Download SFT checkpoint (run once, persists on Drive) ─────────────────
# Starting point for RL fine-tuning.
# Skip if checkpoints/cadrille-sft already exists on Drive.

import os

SFT_CKPT = 'checkpoints/cadrille-sft'
if not os.path.exists(SFT_CKPT):
    from huggingface_hub import snapshot_download
    snapshot_download(
        repo_id='maksimko123/cadrille',
        local_dir=SFT_CKPT)
    print(f'Downloaded → {SFT_CKPT}')
else:
    print(f'Already exists: {SFT_CKPT}')

In [ ]:
# ── [7] W&B login ────────────────────────────────────────────────────────────
import wandb
wandb.login()   # paste your API key from wandb.ai/authorize

In [ ]:
# ── [8] RL fine-tuning on H100 ───────────────────────────────────────────────
# Uses configs/rl/h100.yaml:
#   G=16, top_N=4, batch_updates=3, max_new_tokens=400
#   Validation: 100 DeepCAD + 100 Fusion360, pc + img modalities
#
# Checkpoints are saved to Drive every 1000 steps.
# If the session crashes, re-run this cell — it resumes from the latest checkpoint
# by setting --run-name to the same value (W&B resume='allow').
#
# Expected throughput on H100: ~8-12 steps/min → 50k steps in ~3-4 days.

RUN_NAME = 'cadrille-rl-h100-v1'   # change for a new run; keep same to resume

!python rl/train.py \
    --config  configs/rl/h100.yaml \
    --run-name {RUN_NAME}

In [ ]:
# ── [9] Resume after session timeout ─────────────────────────────────────────
# Colab sessions timeout (~12h Pro, ~24h Pro+).
# When your session restarts:
#   1. Re-run cells [1] – [7] (Drive mount, clone, install, link, login)
#   2. Run this cell to continue from the latest saved checkpoint.
#
# W&B will attach to the same run automatically (resume='allow' is set in train.py).

import glob

RUN_NAME = 'cadrille-rl-h100-v1'   # must match the original run

ckpt_dirs = sorted(glob.glob(f'checkpoints/{RUN_NAME}/checkpoint-[0-9]*'))
if not ckpt_dirs:
    print('No checkpoints found — start a fresh run with cell [8] instead.')
else:
    latest = ckpt_dirs[-1]
    print(f'Resuming from: {latest}')
    !python rl/train.py \
        --config           configs/rl/h100.yaml \
        --run-name         {RUN_NAME} \
        --checkpoint-path  {latest}